In [ ]:
""" Because we are using Gemini (for now), the best way to create an agent is by first creating a model"""

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite") #most efficient and inexpensive for current task

agent = create_agent(model=model) #Then create an agent with the model as that we have created prior

To "call" the agent, we use the invoke function.

A dictionary is passed as the argument which includes{"messages" : Message to be passed}

HumanMessage indicates that the attached message is passed as the user query.

In [2]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is the temperature of the sun's core?")]}
)

We can then print the response

In [4]:
from pprint import pprint
pprint(response)

{'messages': [HumanMessage(content="What is the temperature of the sun's core?", additional_kwargs={}, response_metadata={}, id='0f1a8403-e61d-4970-ba26-ca1aa1ded8e0'),
              AIMessage(content="The temperature of the Sun's core is estimated to be around **15 million degrees Celsius (15 million °C)** or **27 million degrees Fahrenheit (27 million °F)**.\n\nThis incredibly high temperature is what drives nuclear fusion, the process that powers the Sun and releases immense amounts of energy.", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019bbcb4-c913-7da1-b9ec-4f91854e39e9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 65, 'total_tokens': 77, 'input_token_details': {'cache_read': 0}})]}


Notice that we recieve back a dictionary that includes data like what message was sent, token data, parameter data (kwags) e.t.c

We can isolate this using list and dictionary properties.

In [5]:
print(response['messages'][-1].content)

The temperature of the Sun's core is estimated to be around **15 million degrees Celsius (15 million °C)** or **27 million degrees Fahrenheit (27 million °F)**.

This incredibly high temperature is what drives nuclear fusion, the process that powers the Sun and releases immense amounts of energy.


Because we are passing in a dictionary, we could pass in chat history.
Let's see if we can gaslight the model into thinking it gave us back some wrong information.

In [6]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What's the capital of the Moon?"),
    AIMessage(content="The capital of the Moon is Luna City."),
    HumanMessage(content="Interesting, tell me more about Luna City")]}
)

pprint(response)

{'messages': [HumanMessage(content="What's the capital of the Moon?", additional_kwargs={}, response_metadata={}, id='878a65d7-5ae3-4d95-a947-7e0555651020'),
              AIMessage(content='The capital of the Moon is Luna City.', additional_kwargs={}, response_metadata={}, id='66322cab-6c51-4ada-a9a0-4b46dc2e7794', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content='Interesting, tell me more about Luna City', additional_kwargs={}, response_metadata={}, id='f3579cc4-f229-4e16-a327-5dce1ee5c7e6'),
              AIMessage(content='Luna City is a fictional city on the Moon. It is the setting for a number of science fiction stories, including the novel "The Moon is a Harsh Mistress" by Robert A. Heinlein. In the novel, Luna City is a penal colony that has become a thriving metropolis. It is a city of contrasts, with both advanced technology and a rugged frontier spirit. Luna City is a place of both opportunity and danger, and it is a place that has captured the imagi

In [7]:
print(response['messages'][-1].content)

Luna City is a fictional city on the Moon. It is the setting for a number of science fiction stories, including the novel "The Moon is a Harsh Mistress" by Robert A. Heinlein. In the novel, Luna City is a penal colony that has become a thriving metropolis. It is a city of contrasts, with both advanced technology and a rugged frontier spirit. Luna City is a place of both opportunity and danger, and it is a place that has captured the imagination of many readers.


It seems that it can't be duped on this one.

Notice however the long latency time it took to respond due to the length of our message (passing in chat history) about 3.6 seconds.

One of the ways in which we can reduce the percieved(not actual) latency of our system is by streaming tokens to the user as they appear, rather than streaming them all at once.

This can be observed in all major chatbots, think Chatgpt or Gemini responses.

In [ ]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="Tell me all about Luna City, the capital of the Moon")]},
    stream_mode="messages"
):

    # token is a message chunk with token content
    # metadata contains which node produced the token
    
    if token.content:  # Check if there's actual content
        print(token.content, end="", flush=True)  # Print token

To make the system more fit for purpose, one could give it a prompt.

This could either be a few input output examples or the format in which you would want the response to be in. See the examples below.

In [9]:
system_prompt1= """

You are a science fiction writer, create a space capital city at the users request.

User: What is the capital of mars?
Scifi Writer: Marsialis

User: What is the capital of Venus?
Scifi Writer: Venusovia

"""

In [15]:
system_prompt2= """

You are a science fiction writer, create a space capital city at the users request.

Please keep to the below structure.

Name: The name of the capital city

Location: Where it is based

Vibe: 2-3 words to describe its vibe

Economy: Main industries

"""

Input the system prompt as one of the (kwargs) when creating an agent.

In [18]:
agent = create_agent(
    model=model,
    system_prompt=system_prompt1#switch between 1 and 2 to see the difference
)

response =agent.invoke(
    {"messages": [HumanMessage(content="What is the capital of uranus")]}
)

print(response['messages'][1].content)

Scifi Writer: Urania


We could also instead provide an output schema that we would want the agent to follow.

This is particularly useful if we would like to use the outputting schema later on.

In [21]:
from pydantic import BaseModel

class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

agent = create_agent(
    model=model,
    system_prompt="You are a fiction writer, create a capital city at the users request.",
    response_format=CapitalInfo
)

question = HumanMessage(content="What is the capital of the african country Tzibotswa?")

response = agent.invoke(
    {"messages": [question]}
)

response["structured_response"]

CapitalInfo(name='Kibali', location='Central Tzibotswa, on the banks of the Great Rift River', vibe='Bustling and ancient, with a mix of traditional markets and soaring modern architecture.', economy='Heavily reliant on mining of rare earth minerals and a growing tech sector.')

Because of the structured properties, we could then just call a specific output.

In [22]:
response["structured_response"].name

'Kibali'

In [23]:
capital_info = response["structured_response"]

capital_name = capital_info.name
capital_location = capital_info.location

print(f"{capital_name} is a city located at {capital_location}")

Kibali is a city located at Central Tzibotswa, on the banks of the Great Rift River
